# 01 — Instance Parser and ILP

This notebook parses the benchmark instance files and checks them against the published sizes, before the ILP model is built on top of the same parser.

Dataset: Nurse Rostering Benchmark Instances 1-24 (Curtois & Qu, University of Nottingham),
<https://www.schedulingbenchmarks.org/nrp/>. Format guide:
<https://www.schedulingbenchmarks.org/nrp/instances1_24.html>. The parser lives in
`rostering_common.py` at the repository root, so it can be imported unchanged from every notebook
(`01_ilp.ipynb`, `02_ga.ipynb`, `03_comparison.ipynb`).

## File format, section by section

Each `data/raw/InstanceN.txt` is a plain-text file. Lines starting with `#` are comments and blank
lines are ignored; every other line belongs to the section named by the most recent `SECTION_...`
header. Below is what each section means and how it is stored in the `Instance` dataclass.

| Section | Fields (comma-separated) | Meaning | Stored as |
|---|---|---|---|
| `SECTION_HORIZON` | `n_days` | Length of the planning period in days. Day 0 is always a Monday. | `int` — `Instance.n_days` |
| `SECTION_SHIFTS` | `shift_id, length_minutes, forbidden_next` | One shift type: its id (can be several characters, e.g. `d1`, `a2`), its length in minutes, and the `\|`-separated list of shift ids that must **not** be worked the day after this one (the list can be empty, e.g. `D,480,` for a shift with no restriction). | `shift_ids: list[str]` (keeps file order), `shift_len: dict[str,int]`, `forbidden_next: dict[str, set[str]]` |
| `SECTION_STAFF` | `nurse_id, max_shifts, max_minutes, min_minutes, max_consec, min_consec, min_days_off, max_weekends` | One nurse's contract. `max_shifts` is itself a `\|`-separated list `shift=count` (e.g. `E=14\|D=14\|L=0`); a count of 0 means the nurse can never work that shift type (used to encode skills). The rest are single integers: total-minutes window, longest and shortest allowed working block, shortest allowed rest block, and the cap on weekends worked. | `nurse_ids: list[str]`, `max_shifts: dict[str, dict[str,int]]`, and one `dict[str,int]` per remaining field |
| `SECTION_DAYS_OFF` | `nurse_id, day, day, ...` | Days on which this nurse must not be assigned any shift (hard rule). A nurse can have zero listed days, in which case the line for them is simply absent. | `days_off: dict[str, set[int]]` (every nurse gets an entry, even an empty set) |
| `SECTION_SHIFT_ON_REQUESTS` | `nurse_id, day, shift_id, weight` | The nurse asks to work this shift on this day; not granting it costs `weight` in the objective. | `on_requests: list[tuple[str,int,str,int]]` |
| `SECTION_SHIFT_OFF_REQUESTS` | `nurse_id, day, shift_id, weight` | The nurse asks **not** to work this shift on this day; assigning it anyway costs `weight`. | `off_requests: list[tuple[str,int,str,int]]` |
| `SECTION_COVER` | `day, shift_id, required, weight_under, weight_over` | How many nurses are required on this day for this shift, and the penalty per nurse for being short (`weight_under`) or over (`weight_over`). | `cover: dict[tuple[int,str], tuple[int,int,int]]` |

Traps to watch for (all handled by `load_instance`, see `rostering_common.py`): a shift id can be
more than one character; a forbidden-next list can be empty; `max_shifts` entries can appear in any
order and are not necessarily one-per-shift-type; comment and blank lines can appear between any two
data lines, not just at section boundaries.

In [1]:
import pandas as pd

from rostering_common import load_instance

inst1 = load_instance("data/raw/Instance1.txt")
print(f"name           = {inst1.name}")
print(f"n_days         = {inst1.n_days}  (n_weeks = {inst1.n_weeks})")
print(f"shift_ids      = {inst1.shift_ids}")
print(f"n_nurses       = {inst1.n_nurses}")
print(f"nurse_ids      = {inst1.nurse_ids}")
print(f"max_shifts['A']= {inst1.max_shifts['A']}")
print(f"days_off['A']  = {inst1.days_off['A']}")
print(f"cover[(0,'D')] = {inst1.cover[(0, 'D')]}   # (required, weight_under, weight_over)")
print(f"# on-requests  = {len(inst1.on_requests)}, # off-requests = {len(inst1.off_requests)}")


name           = Instance1
n_days         = 14  (n_weeks = 2)
shift_ids      = ['D']
n_nurses       = 8
nurse_ids      = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
max_shifts['A']= {'D': 14}
days_off['A']  = {0}
cover[(0,'D')] = (5, 100, 1)   # (required, weight_under, weight_over)
# on-requests  = 21, # off-requests = 5


## Test: all 24 instances load, and their sizes match `data/best_known.csv`

For every instance, `n_nurses`, `n_weeks` and `n_shift_types` (from the parsed file) must equal the
`employees`, `weeks` and `shift_types` columns of `data/best_known.csv` (the numbers the benchmark
page itself publishes for each instance).

In [2]:
best_known = pd.read_csv("data/best_known.csv").set_index("instance")

rows = []
mismatches = []
for i in range(1, 25):
    name = f"Instance{i}"
    inst = load_instance(f"data/raw/{name}.txt")
    expected = best_known.loc[name]
    ok_nurses = inst.n_nurses == expected["employees"]
    ok_weeks = inst.n_weeks == expected["weeks"]
    ok_shifts = inst.n_shift_types == expected["shift_types"]
    rows.append({
        "instance": name,
        "n_days": inst.n_days,
        "n_weeks": inst.n_weeks,
        "n_nurses": inst.n_nurses,
        "n_shift_types": inst.n_shift_types,
        "n_binaries": inst.n_nurses * inst.n_days * inst.n_shift_types,
        "sizes_match_best_known": ok_nurses and ok_weeks and ok_shifts,
    })
    if not (ok_nurses and ok_weeks and ok_shifts):
        mismatches.append(name)

sizes = pd.DataFrame(rows).set_index("instance")
display(sizes)

print()
assert len(sizes) == 24, f"expected 24 instances, loaded {len(sizes)}"
assert not mismatches, f"size mismatch against best_known.csv for: {mismatches}"
assert sizes["sizes_match_best_known"].all()
print("PASS: 24 of 24 files load, and n_nurses / n_weeks / n_shift_types match data/best_known.csv for every one.")


,n_days,n_weeks,n_nurses,n_shift_types,n_binaries,sizes_match_best_known
instance,,,,,,
Instance1,14,2,8,1,112,True
Instance2,14,2,14,2,392,True
Instance3,14,2,20,3,840,True
Instance4,28,4,10,2,560,True
Instance5,28,4,16,2,896,True
Instance6,28,4,18,3,1512,True
Instance7,28,4,20,3,1680,True
Instance8,28,4,30,4,3360,True
Instance9,28,4,36,4,4032,True



PASS: 24 of 24 files load, and n_nurses / n_weeks / n_shift_types match data/best_known.csv for every one.


## Try to break it — five edge cases

Five small, hand-written instance snippets, each targeting one thing the real files do that a naive
parser could get wrong. Each is written to a temporary file and parsed with the same
`load_instance` used above — no special-casing.

In [3]:
import tempfile
import os

def parse_snippet(text: str):
    """Write text to a temp file and parse it, like a real InstanceN.txt."""
    with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False, encoding="utf-8") as f:
        f.write(text)
        path = f.name
    try:
        return load_instance(path)
    finally:
        os.remove(path)


edge_case_1 = """\
SECTION_HORIZON
7

SECTION_SHIFTS
# a shift with NO forbidden-next entry at all (trailing comma, nothing after it)
D,480,

SECTION_STAFF
A,D=7,3360,3360,7,1,1,1

SECTION_DAYS_OFF

SECTION_SHIFT_ON_REQUESTS

SECTION_SHIFT_OFF_REQUESTS

SECTION_COVER
0,D,1,100,1
"""
inst = parse_snippet(edge_case_1)
assert inst.forbidden_next["D"] == set(), "empty forbidden-next list should parse to an empty set"
assert inst.days_off["A"] == set(), "a nurse with no SECTION_DAYS_OFF line should get an empty set, not a KeyError"
print("1. Empty forbidden-next list + nurse absent from SECTION_DAYS_OFF -> OK")


edge_case_2 = """\
SECTION_HORIZON
7

SECTION_SHIFTS
# multi-letter shift ids, as the format guide allows (e.g. d1, a2)
d1,480,a2
a2,600,d1

SECTION_STAFF
A,d1=5|a2=2,3360,3360,7,1,1,1

SECTION_DAYS_OFF

SECTION_SHIFT_ON_REQUESTS

SECTION_SHIFT_OFF_REQUESTS

SECTION_COVER
0,d1,1,100,1
"""
inst = parse_snippet(edge_case_2)
assert inst.shift_ids == ["d1", "a2"]
assert inst.forbidden_next["d1"] == {"a2"}
assert inst.max_shifts["A"] == {"d1": 5, "a2": 2}
print("2. Multi-letter shift ids (d1, a2) -> OK")


edge_case_3 = """\
SECTION_HORIZON
7

SECTION_SHIFTS
E,480,
D,480,E
L,480,E|D

SECTION_STAFF
# a shift count of 0 (nurse can never work that shift type)
A,E=7|D=0|L=0,3360,3360,7,1,1,1

SECTION_DAYS_OFF

SECTION_SHIFT_ON_REQUESTS

SECTION_SHIFT_OFF_REQUESTS

SECTION_COVER
0,E,1,100,1
"""
inst = parse_snippet(edge_case_3)
assert inst.max_shifts["A"]["D"] == 0 and inst.max_shifts["A"]["L"] == 0
print("3. MaxShifts entry of 0 for several shift types (E=7|D=0|L=0) -> OK")


edge_case_4 = """\
SECTION_HORIZON
# a comment line dropped in between two data lines of the same section
7

SECTION_SHIFTS
D,480,

SECTION_STAFF
A,D=7,3360,3360,7,1,1,1
# a comment between two staff lines
B,D=7,3360,3360,7,1,1,1

SECTION_DAYS_OFF
A,2,5
# comment
B,0

SECTION_SHIFT_ON_REQUESTS

SECTION_SHIFT_OFF_REQUESTS

SECTION_COVER
0,D,2,100,1
"""
inst = parse_snippet(edge_case_4)
assert inst.nurse_ids == ["A", "B"]
assert inst.days_off["A"] == {2, 5} and inst.days_off["B"] == {0}
print("4. Comment lines interleaved between data lines within a section -> OK")


edge_case_5 = """\
SECTION_HORIZON
7

SECTION_SHIFTS
E,480,
D,480,E
L,480,E|D

SECTION_STAFF
A,E=7|D=7|L=7,3360,3360,7,1,1,1

SECTION_DAYS_OFF

SECTION_SHIFT_ON_REQUESTS
A,1,E,3
A,3,L,1

SECTION_SHIFT_OFF_REQUESTS
A,5,D,2

SECTION_COVER
0,E,1,100,1
0,D,1,100,1
0,L,1,100,1
"""
inst = parse_snippet(edge_case_5)
assert inst.on_requests == [("A", 1, "E", 3), ("A", 3, "L", 1)]
assert inst.off_requests == [("A", 5, "D", 2)]
assert len(inst.cover) == 3
print("5. Several shift types with both on- and off-requests, and one cover row per shift -> OK")

print()
print("All 5 edge cases handled correctly by the same load_instance() used on the real 24 files.")


1. Empty forbidden-next list + nurse absent from SECTION_DAYS_OFF -> OK
2. Multi-letter shift ids (d1, a2) -> OK
3. MaxShifts entry of 0 for several shift types (E=7|D=0|L=0) -> OK
4. Comment lines interleaved between data lines within a section -> OK
5. Several shift types with both on- and off-requests, and one cover row per shift -> OK

All 5 edge cases handled correctly by the same load_instance() used on the real 24 files.


---

## ILP model v1 — basic rules and cover (C1–C4, C9, C10)

### Variables
- $x_{idt}\in\{0,1\}$ — 1 if nurse $i$ works shift $t$ on day $d$.
- $u_{dt},v_{dt}\ge 0$ — shortfall (under-cover) and surplus (over-cover) on day $d$, shift $t$.

### Objective (minimise)
$$\sum_{d,t} \big(w^{-}_{dt} u_{dt} + w^{+}_{dt} v_{dt}\big) \;+\; \sum_{(i,d,t)\in \text{on-requests}} w_{idt}(1-x_{idt}) \;+\; \sum_{(i,d,t)\in \text{off-requests}} w_{idt}\, x_{idt}$$

### Constraints in this first pass
- **C1** (one shift per day): $\sum_t x_{idt} \le 1$
- **C2** (forbidden successions): $x_{idt} + x_{i,d+1,u} \le 1$ for every $u$ in shift $t$'s forbidden-next list
- **C3** (max shifts per type): $\sum_d x_{idt} \le M_{it}$
- **C4** (working-time window): $L^{\min}_i \le \sum_{d,t} \ell_t x_{idt} \le L^{\max}_i$
- **C9** (fixed days off): $x_{idt}=0$ for $d\in O_i$
- **C10** (cover balance): $\sum_i x_{idt} + u_{dt} - v_{dt} = r_{dt}$

**Not included yet** (Day 3): C5 max consecutive working days, C6 min consecutive working days, C7 min
consecutive days off, C8 max weekends. Because this first pass has *fewer* restrictions than the full
model, its optimal value can only be **equal to or lower** than the published best-known value — it is a
relaxation, not yet the full problem. Reproducing 607 (Instance 1) and 828 (Instance 2) exactly is the
acceptance test once C5–C8 are added.

In [4]:
import pulp


def solve_ilp_v1(inst, time_limit=120, msg=False):
    """Build and solve the C1-C4, C9, C10 model for one Instance. Returns a dict with the
    objective, solver status, and the raw PuLP problem (for inspecting the roster if needed)."""
    nurses, shifts, D = inst.nurse_ids, inst.shift_ids, range(inst.n_days)

    prob = pulp.LpProblem(inst.name, pulp.LpMinimize)

    x = pulp.LpVariable.dicts("x", (nurses, D, shifts), cat="Binary")
    under = pulp.LpVariable.dicts("under", list(inst.cover), lowBound=0)
    over = pulp.LpVariable.dicts("over", list(inst.cover), lowBound=0)

    # Objective
    obj_terms = []
    for (d, t), (required, w_under, w_over) in inst.cover.items():
        obj_terms.append(w_under * under[(d, t)] + w_over * over[(d, t)])
    for (i, d, t, w) in inst.on_requests:
        obj_terms.append(w * (1 - x[i][d][t]))
    for (i, d, t, w) in inst.off_requests:
        obj_terms.append(w * x[i][d][t])
    prob += pulp.lpSum(obj_terms)

    # C10: cover balance (only for the (day, shift) pairs the instance actually specifies)
    for (d, t), (required, w_under, w_over) in inst.cover.items():
        prob += pulp.lpSum(x[i][d][t] for i in nurses) + under[(d, t)] - over[(d, t)] == required

    for i in nurses:
        # C1: at most one shift per day
        for d in D:
            prob += pulp.lpSum(x[i][d][t] for t in shifts) <= 1

        # C2: forbidden successions
        for d in range(inst.n_days - 1):
            for t in shifts:
                for u in inst.forbidden_next[t]:
                    prob += x[i][d][t] + x[i][d + 1][u] <= 1

        # C3: max shifts of each type
        for t in shifts:
            prob += pulp.lpSum(x[i][d][t] for d in D) <= inst.max_shifts[i].get(t, 0)

        # C4: working-time window
        minutes = pulp.lpSum(inst.shift_len[t] * x[i][d][t] for d in D for t in shifts)
        prob += minutes >= inst.min_minutes[i]
        prob += minutes <= inst.max_minutes[i]

        # C9: fixed days off
        for d in inst.days_off[i]:
            for t in shifts:
                prob += x[i][d][t] == 0

    solver = pulp.PULP_CBC_CMD(msg=msg, timeLimit=time_limit)
    prob.solve(solver)

    # Trap: LpStatus can say "Optimal" even when CBC actually stopped on the time limit.
    # prob.sol_status is the reliable one: 1 = optimal, 2 = feasible (not proven optimal).
    status_text = {1: "proven optimal", 2: "feasible (time limit reached)"}.get(prob.sol_status, "unknown")

    return {
        "instance": inst.name,
        "objective": pulp.value(prob.objective),
        "status_text": status_text,
        "sol_status": prob.sol_status,
        "problem": prob,
    }


### Run on Instance 1 and Instance 2, compare with `best_known.csv`

In [5]:
results_v1 = []
for name in ["Instance1", "Instance2"]:
    inst = load_instance(f"data/raw/{name}.txt")
    r = solve_ilp_v1(inst, time_limit=120)
    expected = best_known.loc[name, "best_known_solution"]
    results_v1.append({
        "instance": name,
        "objective_v1": r["objective"],
        "status": r["status_text"],
        "best_known": expected,
        "gap_to_best_known": r["objective"] - expected,
    })
    print(f"{name}: objective = {r['objective']:.0f}  ({r['status_text']}),  "
          f"best-known = {expected},  difference = {r['objective'] - expected:+.0f}")

pd.DataFrame(results_v1).set_index("instance")


Instance1: objective = 3  (proven optimal),  best-known = 607,  difference = -604


Instance2: objective = 208  (proven optimal),  best-known = 828,  difference = -620


,objective_v1,status,best_known,gap_to_best_known
instance,,,,
Instance1,3.0,proven optimal,607,-604.0
Instance2,208.0,proven optimal,828,-620.0


A difference of 0 would mean this first-pass model already matches the full best-known value on
these two instances — meaning C5–C8 happen not to bind for them. A negative difference (lower than
best-known) would confirm this is a true relaxation on at least one instance: the model is finding a
cheaper roster that the full hard-rule set (consecutive-day / weekend limits) would actually forbid.
Either way, the real acceptance test is Day 3's, once C5–C8 are added.

---

## ILP model v2 — the remaining hard rules (C5–C8), plus the LP relaxation

### The four rules this adds

- **C5** (max consecutive working days): any $c^{\max}_i+1$ consecutive days must contain at least one
  day off.
- **C6** (min consecutive working days): a working block shorter than $c^{\min}_i$ days is forbidden —
  no lone single working days wedged between rest days.
- **C7** (min consecutive days off): a rest block shorter than $o^{\min}_i$ days is forbidden.
- **C8** (max weekends): a weekend counts as worked if a nurse is given a shift on the Saturday or the
  Sunday of that week; each nurse may work at most $a^{\max}_i$ weekends over the whole horizon.

**Boundary rule:** the day before day 0, and the day after the last day, are both treated as a day off.
A working or resting block that touches either end of the horizon has an unknown true length, so C6 and
C7 only check blocks that are fully *inside* the horizon — this is already how the day ranges below are
written (they never look before day 0 or after the last day).

With C1–C10 all included, this is now the *complete* model from the problem statement — the acceptance
test below is whether it reproduces the benchmark's own proven optimal values exactly.

In [6]:
def solve_ilp(inst, time_limit=120, msg=False):
    """Full C1-C10 model for one Instance. Same as solve_ilp_v1, plus C5-C8."""
    nurses, shifts, D, h = inst.nurse_ids, inst.shift_ids, range(inst.n_days), inst.n_days

    prob = pulp.LpProblem(inst.name, pulp.LpMinimize)

    x = pulp.LpVariable.dicts("x", (nurses, D, shifts), cat="Binary")
    under = pulp.LpVariable.dicts("under", list(inst.cover), lowBound=0)
    over = pulp.LpVariable.dicts("over", list(inst.cover), lowBound=0)

    def w(i, d):
        """Shorthand: 1 if nurse i works any shift on day d (sum is 0 or 1 because of C1)."""
        return pulp.lpSum(x[i][d][t] for t in shifts)

    # Objective (same as v1)
    obj_terms = []
    for (d, t), (required, w_under, w_over) in inst.cover.items():
        obj_terms.append(w_under * under[(d, t)] + w_over * over[(d, t)])
    for (i, d, t, wt) in inst.on_requests:
        obj_terms.append(wt * (1 - x[i][d][t]))
    for (i, d, t, wt) in inst.off_requests:
        obj_terms.append(wt * x[i][d][t])
    prob += pulp.lpSum(obj_terms)

    # C10: cover balance
    for (d, t), (required, w_under, w_over) in inst.cover.items():
        prob += pulp.lpSum(x[i][d][t] for i in nurses) + under[(d, t)] - over[(d, t)] == required

    n_weeks = h // 7
    for i in nurses:
        for d in D:                                                              # C1
            prob += pulp.lpSum(x[i][d][t] for t in shifts) <= 1

        for d in range(h - 1):                                                   # C2
            for t in shifts:
                for u in inst.forbidden_next[t]:
                    prob += x[i][d][t] + x[i][d + 1][u] <= 1

        for t in shifts:                                                         # C3
            prob += pulp.lpSum(x[i][d][t] for d in D) <= inst.max_shifts[i].get(t, 0)

        minutes = pulp.lpSum(inst.shift_len[t] * x[i][d][t] for d in D for t in shifts)
        prob += minutes >= inst.min_minutes[i]                                   # C4
        prob += minutes <= inst.max_minutes[i]

        max_c = inst.max_consec[i]                                               # C5
        for d in range(h - max_c):
            prob += pulp.lpSum(w(i, j) for j in range(d, d + max_c + 1)) <= max_c

        min_c = inst.min_consec[i]                                               # C6
        for s in range(1, min_c):
            for d in range(h - s - 1):
                prob += w(i, d) + (s - pulp.lpSum(w(i, j) for j in range(d + 1, d + s + 1))) \
                        + w(i, d + s + 1) >= 1

        min_off = inst.min_days_off[i]                                           # C7
        for s in range(1, min_off):
            for d in range(h - s - 1):
                prob += (1 - w(i, d)) + pulp.lpSum(w(i, j) for j in range(d + 1, d + s + 1)) \
                        + (1 - w(i, d + s + 1)) >= 1

        k = pulp.LpVariable.dicts(f"k_{i}", range(n_weeks), cat="Binary")        # C8
        for e in range(n_weeks):
            prob += w(i, 7 * e + 5) + w(i, 7 * e + 6) <= 2 * k[e]
        prob += pulp.lpSum(k[e] for e in range(n_weeks)) <= inst.max_weekends[i]

        for d in inst.days_off[i]:                                               # C9
            for t in shifts:
                prob += x[i][d][t] == 0

    solver = pulp.PULP_CBC_CMD(msg=msg, timeLimit=time_limit)
    prob.solve(solver)

    status_text = {1: "proven optimal", 2: "feasible (time limit reached)"}.get(prob.sol_status, "unknown")
    return {
        "instance": inst.name,
        "objective": pulp.value(prob.objective),
        "status_text": status_text,
        "sol_status": prob.sol_status,
        "problem": prob,
    }


### Acceptance test — must reproduce the proven optimal values exactly

In [7]:
acceptance = []
for name in ["Instance1", "Instance2"]:
    inst = load_instance(f"data/raw/{name}.txt")
    r = solve_ilp(inst, time_limit=120)
    expected = int(best_known.loc[name, "best_known_solution"])
    match = int(r["objective"]) == expected
    acceptance.append({"instance": name, "objective": r["objective"], "status": r["status_text"],
                        "best_known": expected, "matches_exactly": match})
    print(f"{name}: objective = {r['objective']:.0f}  ({r['status_text']}),  "
          f"best-known = {expected},  exact match = {match}")

acc_df = pd.DataFrame(acceptance).set_index("instance")
assert acc_df["matches_exactly"].all(), "the full model must reproduce the proven optimal values exactly"
print()
print("PASS: the full C1-C10 model reproduces the proven optimal values exactly on Instance 1 and 2.")
acc_df


Instance1: objective = 607  (proven optimal),  best-known = 607,  exact match = True


Instance2: objective = 828  (proven optimal),  best-known = 828,  exact match = True

PASS: the full C1-C10 model reproduces the proven optimal values exactly on Instance 1 and 2.


,objective,status,best_known,matches_exactly
instance,,,,
Instance1,607.0,proven optimal,607,True
Instance2,828.0,proven optimal,828,True


### Instance 3 — a larger, three-shift-type instance

Instance 3 (20 nurses, 3 shift types) is harder to prove than Instance 1/2 within a short time limit.
A separate, longer check (outside this notebook) previously reached the published best-known value of
1001 after about 400 seconds without fully proving it. Here we run a shorter limit and report whatever
status CBC actually reaches honestly, rather than assuming it proves optimality.

In [8]:
inst3 = load_instance("data/raw/Instance3.txt")
r3 = solve_ilp(inst3, time_limit=90)
expected3 = int(best_known.loc["Instance3", "best_known_solution"])

print(f"Instance3: objective = {r3['objective']:.0f}  ({r3['status_text']})")
print(f"best-known (proven) = {expected3}")
print(f"gap to best-known   = {r3['objective'] - expected3:+.0f}")
if r3["sol_status"] != 1:
    print()
    print("Not proven optimal within this time limit -- report this run as 'feasible, best found "
          "so far', not as a proven optimum. A longer time budget (minutes, not seconds) would be "
          "needed to close the remaining gap, consistent with the literature on this benchmark.")


Instance3: objective = 1003  (feasible (time limit reached))
best-known (proven) = 1001
gap to best-known   = +2

Not proven optimal within this time limit -- report this run as 'feasible, best found so far', not as a proven optimum. A longer time budget (minutes, not seconds) would be needed to close the remaining gap, consistent with the literature on this benchmark.


### LP relaxation — a lower bound with no Branch and Bound at all

Allow every $x_{idt}$ and $k_{ie}$ to be any fraction between 0 and 1 instead of only 0 or 1, and solve
the resulting ordinary Linear Program. Because every real (integer) roster is also a valid fractional
solution, the relaxed optimum can only be the same as or *lower* than the true integer optimum — it is
a guaranteed lower bound, computed without any branching at all.

In [9]:
def solve_lp_relaxation(inst, msg=False):
    """Same model as solve_ilp, but every variable is continuous on [0, 1] instead of binary."""
    nurses, shifts, D, h = inst.nurse_ids, inst.shift_ids, range(inst.n_days), inst.n_days

    prob = pulp.LpProblem(inst.name + "_relax", pulp.LpMinimize)
    x = pulp.LpVariable.dicts("x", (nurses, D, shifts), lowBound=0, upBound=1)
    under = pulp.LpVariable.dicts("under", list(inst.cover), lowBound=0)
    over = pulp.LpVariable.dicts("over", list(inst.cover), lowBound=0)

    def w(i, d):
        return pulp.lpSum(x[i][d][t] for t in shifts)

    obj_terms = []
    for (d, t), (required, w_under, w_over) in inst.cover.items():
        obj_terms.append(w_under * under[(d, t)] + w_over * over[(d, t)])
    for (i, d, t, wt) in inst.on_requests:
        obj_terms.append(wt * (1 - x[i][d][t]))
    for (i, d, t, wt) in inst.off_requests:
        obj_terms.append(wt * x[i][d][t])
    prob += pulp.lpSum(obj_terms)

    for (d, t), (required, w_under, w_over) in inst.cover.items():
        prob += pulp.lpSum(x[i][d][t] for i in nurses) + under[(d, t)] - over[(d, t)] == required

    n_weeks = h // 7
    for i in nurses:
        for d in D:
            prob += pulp.lpSum(x[i][d][t] for t in shifts) <= 1
        for d in range(h - 1):
            for t in shifts:
                for u in inst.forbidden_next[t]:
                    prob += x[i][d][t] + x[i][d + 1][u] <= 1
        for t in shifts:
            prob += pulp.lpSum(x[i][d][t] for d in D) <= inst.max_shifts[i].get(t, 0)
        minutes = pulp.lpSum(inst.shift_len[t] * x[i][d][t] for d in D for t in shifts)
        prob += minutes >= inst.min_minutes[i]
        prob += minutes <= inst.max_minutes[i]
        max_c = inst.max_consec[i]
        for d in range(h - max_c):
            prob += pulp.lpSum(w(i, j) for j in range(d, d + max_c + 1)) <= max_c
        min_c = inst.min_consec[i]
        for s in range(1, min_c):
            for d in range(h - s - 1):
                prob += w(i, d) + (s - pulp.lpSum(w(i, j) for j in range(d + 1, d + s + 1))) \
                        + w(i, d + s + 1) >= 1
        min_off = inst.min_days_off[i]
        for s in range(1, min_off):
            for d in range(h - s - 1):
                prob += (1 - w(i, d)) + pulp.lpSum(w(i, j) for j in range(d + 1, d + s + 1)) \
                        + (1 - w(i, d + s + 1)) >= 1
        k = pulp.LpVariable.dicts(f"k_{i}", range(n_weeks), lowBound=0, upBound=1)
        for e in range(n_weeks):
            prob += w(i, 7 * e + 5) + w(i, 7 * e + 6) <= 2 * k[e]
        prob += pulp.lpSum(k[e] for e in range(n_weeks)) <= inst.max_weekends[i]
        for d in inst.days_off[i]:
            for t in shifts:
                prob += x[i][d][t] == 0

    prob.solve(pulp.PULP_CBC_CMD(msg=msg))
    return pulp.value(prob.objective)


lp_rows = []
for name in ["Instance1", "Instance2"]:
    inst = load_instance(f"data/raw/{name}.txt")
    f_lp = solve_lp_relaxation(inst)
    f_ilp = int(best_known.loc[name, "best_known_solution"])
    lp_rows.append({"instance": name, "LP_relaxation": f_lp, "ILP_optimum": f_ilp,
                     "gap": f_ilp - f_lp, "gap_pct": (f_ilp - f_lp) / f_ilp * 100 if f_ilp else 0})
    print(f"{name}: LP relaxation = {f_lp:.2f}   ILP optimum = {f_ilp}   "
          f"gap = {f_ilp - f_lp:.2f} ({(f_ilp - f_lp) / f_ilp * 100:.1f}% of the optimum)")

pd.DataFrame(lp_rows).set_index("instance")


Instance1: LP relaxation = 405.00   ILP optimum = 607   gap = 202.00 (33.3% of the optimum)
Instance2: LP relaxation = 717.33   ILP optimum = 828   gap = 110.67 (13.4% of the optimum)


,LP_relaxation,ILP_optimum,gap,gap_pct
instance,,,,
Instance1,405.000004,607,201.999996,33.278418
Instance2,717.333334,828,110.666666,13.365539


A small gap here means the LP relaxation is already a tight, useful bound for Branch and Bound to
prune against — part of why CBC proves Instance 1 and 2 so quickly (well under a second to a few
seconds) despite the astronomically large number of possible rosters.

## Status

- [x] `rostering_common.py` created with `load_instance(path) -> Instance`.
- [x] All 24 of 24 files load; `n_nurses` / `n_weeks` / `n_shift_types` match `data/best_known.csv` for every one.
- [x] Five edge cases (empty forbidden-next, multi-letter shift ids, a zero `MaxShifts` entry,
      interleaved comments, multiple requests/cover rows) parse correctly.
- [x] ILP model v1 (C1-C4, C9, C10) built and solved on Instance 1 and Instance 2.
- [x] Full ILP model (C1-C10) built; acceptance test passes — reproduces 607 (Instance 1) and
      828 (Instance 2) exactly.
- [x] Instance 3 run with a shorter time limit, reported honestly as feasible/not-yet-proven.
- [x] LP relaxation computed for Instance 1 and 2, compared against the proven optimum.

Next (Day 4): run the full model on Instances 1, 2, 3, 6, 8, 12 with a 300 s time limit each,
and save the results to `results/ilp_results.csv`.